In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("TON_IoT_NLP_Dataset.csv")

texts = df['text'].astype(str)
labels = df['label'].astype(int)

print(df.shape)
df.head()


(211043, 3)


,text,label,type
0,tcp protocol connection state oth high duratio...,1,backdoor
1,tcp protocol connection state rej low duration...,1,backdoor
2,tcp protocol connection state rej low duration...,1,backdoor
3,tcp protocol connection state rej low duration...,1,backdoor
4,tcp protocol connection state rej low duration...,1,backdoor


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


#Bi LSTM

In [3]:
#Bi Lstm

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model_bilstm = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_bilstm.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

model_bilstm.summary()


c:\Users\LOQ\miniconda3\envs\myenv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
history_bilstm = model_bilstm.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=256,
    verbose=1
)


Epoch 1/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 59s 96ms/step - accuracy: 0.9240 - loss: 0.1746 - val_accuracy: 0.9372 - val_loss: 0.1379
Epoch 2/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 82s 95ms/step - accuracy: 0.9380 - loss: 0.1407 - val_accuracy: 0.9376 - val_loss: 0.1363
Epoch 3/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 50s 83ms/step - accuracy: 0.9383 - loss: 0.1387 - val_accuracy: 0.9360 - val_loss: 0.1357
Epoch 4/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 53s 89ms/step - accuracy: 0.9390 - loss: 0.1381 - val_accuracy: 0.9365 - val_loss: 0.1348
Epoch 5/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 50s 84ms/step - accuracy: 0.9393 - loss: 0.1368 - val_accuracy: 0.9386 - val_loss: 0.1348


In [6]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred_prob = model_bilstm.predict(X_test_pad).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))


1320/1320 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step
              precision    recall  f1-score   support

           0     0.8936    0.8429    0.8675     10000
           1     0.9521    0.9688    0.9604     32209

    accuracy                         0.9390     42209
   macro avg     0.9228    0.9059    0.9139     42209
weighted avg     0.9382    0.9390    0.9384     42209

ROC-AUC: 0.9827811062125493


#Bi GRRU

In [8]:
from tensorflow.keras.layers import GRU

model_bigru = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(GRU(64, return_sequences=False)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_bigru.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)

model_bigru.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history_bigru = model_bigru.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=256,
    verbose=1
)


Epoch 1/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 54s 87ms/step - accuracy: 0.9198 - loss: 0.1866 - val_accuracy: 0.9361 - val_loss: 0.1455
Epoch 2/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 56s 94ms/step - accuracy: 0.9377 - loss: 0.1420 - val_accuracy: 0.9385 - val_loss: 0.1403
Epoch 3/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 50s 85ms/step - accuracy: 0.9387 - loss: 0.1391 - val_accuracy: 0.9384 - val_loss: 0.1351
Epoch 4/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 53s 88ms/step - accuracy: 0.9386 - loss: 0.1373 - val_accuracy: 0.9391 - val_loss: 0.1456
Epoch 5/5
594/594 ━━━━━━━━━━━━━━━━━━━━ 51s 85ms/step - accuracy: 0.9393 - loss: 0.1361 - val_accuracy: 0.9388 - val_loss: 0.1339


In [10]:
y_pred_prob = model_bigru.predict(X_test_pad).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))


1320/1320 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step
              precision    recall  f1-score   support

           0     0.8928    0.8445    0.8680     10000
           1     0.9525    0.9685    0.9605     32209

    accuracy                         0.9391     42209
   macro avg     0.9227    0.9065    0.9142     42209
weighted avg     0.9384    0.9391    0.9385     42209

ROC-AUC: 0.982703669781738


#BERT